# Unify and consolidate all data

In [ ]:
import pandas as pd
import json
import os
import numpy as np

In [ ]:
SAVE_DIR = "./../data/datasets/"

In [ ]:
def unify_dataset_schema(file_json, file_csv, dataset):
    # create df from json
    df1 = pd.read_json(file_json)
    df1 = df1.transpose()
    df2 = pd.read_csv(file_csv)
    # Initialize the new columns
    df1["Indication_approved_extracted"] = None
    df1["Indication_requested_extracted"] = None
    df1["Marketing_authorisation_holder_extracted"] = None
    
    for row in df1.iterrows():
        document_name = row[1].get("Document_name")
        if document_name in df2["Document_name"].values:
            matching_rows = df2[df2["Document_name"] == document_name]
            if not matching_rows.empty:
                matching_row = matching_rows.iloc[0]
                
                df1.loc[row[0], "Indication_approved_extracted"] = matching_row.get("Indication_approved_extracted", None)
                df1.loc[row[0], "Indication_requested_extracted"] = matching_row.get("Indication_requested_extracted", None)
                df1.loc[row[0], "Marketing_authorisation_holder_extracted"] = matching_row.get("Marketing_authorisation_holder_extracted", None)
                df1.loc[row[0], "Dataset"] = dataset
            else:
                # No matching rows found
                df1.loc[row[0], "Indication_approved_extracted"] = None
                df1.loc[row[0], "Indication_requested_extracted"] = None
                df1.loc[row[0], "Marketing_authorisation_holder_extracted"] = None
        else:
            # Document_name not in df2
            df1.loc[row[0], "Indication_approved_extracted"] = None
            df1.loc[row[0], "Indication_requested_extracted"] = None
            df1.loc[row[0], "Marketing_authorisation_holder_extracted"] = None

    df1 = df1.reindex(sorted(df1.columns), axis=1)

    return df1

## EMA

In [ ]:
filepath_json = "./../data/datasets/manually_cleaned/EMA_manually_cleaned.json"
filepath_csv = "./../data/datasets/with_extracted_data/Diseases_manually_cleaned/EMA_manually_cleaned.csv"
merged_EMA = unify_dataset_schema(filepath_json, filepath_csv, dataset="EMA")
merged_EMA.rename(columns={"Indication_approved_extracted": "Disease_name(s)"}, inplace=True)
merged_EMA.to_csv(SAVE_DIR + "EMA.csv", index=False)
with open(SAVE_DIR + "EMA.json", "w", encoding="utf-8") as out:
    json.dump(merged_EMA.to_dict(orient="index"), out, indent=4, sort_keys=True)


## Swissmedic

In [ ]:
filepath_json = "./../data/datasets/manually_cleaned/SWISSMEDIC_manually_cleaned.json"
filepath_csv = "./../data/datasets/with_extracted_data/Diseases_manually_cleaned/SWISSMEDIC_manually_cleaned.csv"
merged_SWISSMEDIC = unify_dataset_schema(filepath_json, filepath_csv, dataset="Swissmedic")
merged_SWISSMEDIC.rename(columns={"Indication_approved_extracted": "Disease_name(s)"}, inplace=True)
merged_SWISSMEDIC.to_csv(SAVE_DIR + "SWISSMEDIC.csv", index=False)
with open(SAVE_DIR + "SWISSMEDIC.json", "w", encoding="utf-8") as out:
    json.dump(merged_SWISSMEDIC.to_dict(orient="index"), out, indent=4, sort_keys=True)


## Japan

In [ ]:
filepath_json = "./../data/datasets/manually_cleaned/JAPAN_manually_cleaned.json"
filepath_csv = "./../data/datasets/with_extracted_data/Diseases_manually_cleaned/JAPAN_manually_cleaned.csv"
merged_JAPAN = unify_dataset_schema(filepath_json, filepath_csv, dataset="PMDA")
merged_JAPAN["Marketing_authorisation_number"] = merged_JAPAN["Document_name"].str.strip(".pdf")
merged_JAPAN.rename(columns={"Indication_approved_extracted": "Disease_name(s)"}, inplace=True)
merged_JAPAN.to_csv(SAVE_DIR + "JAPAN.csv", index=False)
with open(SAVE_DIR + "JAPAN.json", "w", encoding="utf-8") as out:
    json.dump(merged_JAPAN.to_dict(orient="index"), out, indent=4, sort_keys=True)


In [ ]:
merged_JAPAN.head()

## Australia

In [ ]:
filepath_json = "./../data/datasets/manually_cleaned/AUSTRALIA_manually_cleaned.json"
filepath_csv = "./../data/datasets/with_extracted_data/Diseases_manually_cleaned/AUSTRALIA_manually_cleaned.csv"
merged_AUSTRALIA = unify_dataset_schema(filepath_json, filepath_csv, dataset="TGA")
merged_AUSTRALIA.rename(columns={"Indication_approved_extracted": "Disease_name(s)"}, inplace=True)
merged_AUSTRALIA.to_csv(SAVE_DIR + "AUSTRALIA.csv", index=False)
with open(SAVE_DIR + "AUSTRALIA.json", "w", encoding="utf-8") as out:
    json.dump(merged_AUSTRALIA.to_dict(orient="index"), out, indent=4, sort_keys=True)


## FDA

In [ ]:
with open("./../data/datasets/manually_cleaned/FDA_manually_cleaned.json", "r") as f1:
    data1 = json.load(f1)
with open("./../data/FDA/with_extracted_data_drug_class/FDA.json", "r") as f2:
    data2 = json.load(f2)
with open("./../data/FDA/with_extracted_data_disease_class/FDA.json", "r") as f3:
    data3 = json.load(f3)

# Create lookup for drug class
drug_class_lookup = {}
for entry in data2.values():
    ma_number = entry.get("MA_Number")
    drug_class = entry.get("Non_proprietary_name_extracted")
    if ma_number:
        drug_class_lookup[ma_number] = drug_class

# Create lookup for disease extraction
disease_lookup = {}
for entry in data3.values():
    ma_number = entry.get("MA_Number")
    if ma_number:
        disease_lookup[ma_number] = {
            "disease_class": entry.get("Indications_and_usage_disease_class_extracted"),
            "disease_name": entry.get("Indications_and_usage_disease_name_extracted")
        }

# Enrich data1
for key, entry in data1.items():
    ma_number = entry.get("Marketing_authorisation_number")
    
    # Add drug class
    if ma_number and ma_number in drug_class_lookup:
        entry["Drug_class"] = drug_class_lookup[ma_number]
    else:
        entry["Drug_class"] = None
    
    # Add disease extraction data
    if ma_number and ma_number in disease_lookup:
        entry["Disease_class(es)"] = disease_lookup[ma_number]["disease_class"]
        entry["Indication_approved_extracted"] = disease_lookup[ma_number]["disease_name"]
    else:
        entry["Disease_class(es)"] = None
        entry["Indication_approved_extracted"] = None
    
    # Add standard fields
    entry["Application_date"] = None
    entry["Application_year"] = None
    entry["Document_name"] = None
    entry["Indication_approved"] = entry.get("Indications_and_usage")
    entry.pop("Indications_and_usage", None) 
    entry["Indication_requested"] = None
    entry["Indication_requested_extracted"] = None
    entry["Procedure_number"] = None
    entry["Referral_body"] = entry.get("Referral")
    entry.pop("Referral", None)
    entry["Dataset"] = "FDA"
    entry.pop("Origin", None)
    
    if not ma_number:
        print(f"MA_Number not found for entry: {entry}")

# Rename column in dictionary
for key, entry in data1.items():
    if "Indication_approved_extracted" in entry:
        entry["Disease_name(s)"] = entry.pop("Indication_approved_extracted")
 
# Save to new JSON file
output_path = "./../data/datasets/FDA.json"
with open(output_path, "w", encoding="utf-8") as out:
    json.dump(data1, out, indent=4, sort_keys=True)

print(f"Enriched file saved to {output_path}")

# Save as CSV just in case
df = pd.DataFrame(data1).transpose()
df.to_csv("./../data/datasets/FDA.csv", encoding="utf-8")


In [ ]:
df.columns

## HealthCanada

In [ ]:
man_cleaned_path = "./../data/datasets/manually_cleaned/HEALTHCANADA_manually_cleaned.json"
with_add_columns = "./../data/HealthCanada/with_extracted_data_pdfs_disease_name_class/HEALTHCANADA.json"
pdf_dict_path = "./../data/HealthCanada/pdf_dict.json"

with open(man_cleaned_path, "r") as f1:
    data1 = json.load(f1)
with open(with_add_columns, "r") as f2:
    data2 = json.load(f2)
with open(pdf_dict_path, "r") as f3:
    pdf_dict = json.load(f3)

# Create reverse mapping: MA_number -> PDF_digits
ma_to_pdf = {}
for pdf_url, ma_numbers in pdf_dict.items():
    # Extract digits from PDF URL (e.g., "00078801" from "https://pdf.hres.ca/dpd_pm/00078801.PDF")
    pdf_digits = pdf_url.split('/')[-1].replace('.PDF', '')
    for ma_num in ma_numbers:
        ma_to_pdf[ma_num] = pdf_digits

# Enrich data1 using Marketing_authorisation_number -> PDF digits -> extracted data
for key, entry in data1.items():
    ma_number = entry.get("Marketing_authorisation_number")
    if ma_number and ma_number in ma_to_pdf:
        pdf_digits = ma_to_pdf[ma_number]
        if pdf_digits in data2:
            disease_class = data2[pdf_digits].get("Indications_and_usage_disease_class_extracted")
            entry["Disease_class(es)"] = disease_class
            
            indications_approved = data2[pdf_digits].get("Indications_and_usage")
            entry["Indication_approved"] = indications_approved
            
            # Use disease name extracted for BOTH approved and requested
            disease_name_extracted = data2[pdf_digits].get("Indications_and_usage_disease_name_extracted")
            entry["Indication_approved_extracted"] = disease_name_extracted
            entry["Indication_requested_extracted"] = disease_name_extracted
        else:
            entry["Disease_class(es)"] = None
            entry["Indication_approved"] = None
            entry["Indication_approved_extracted"] = None
            entry["Indication_requested_extracted"] = None
    else:
        entry["Disease_class(es)"] = None
        entry["Indication_approved"] = None
        entry["Indication_approved_extracted"] = None
        entry["Indication_requested_extracted"] = None
    
    entry["Dataset"] = "HealthCanada"

# Add missing columns for consistency with other datasets
for key, entry in data1.items():
    if "Indication_requested" not in entry:
        entry["Indication_requested"] = None

# Rename column in dictionary
for key, entry in data1.items():
    if "Indication_approved_extracted" in entry:
        entry["Disease_name(s)"] = entry.pop("Indication_approved_extracted")

output_path = "./../data/datasets/HEALTHCANADA.json"
with open(output_path, "w", encoding="utf-8") as out:
    json.dump(data1, out, indent=4, sort_keys=True)

# Save as CSV
df = pd.DataFrame(data1).transpose()
df = df.reindex(sorted(df.columns), axis=1)
df.to_csv(SAVE_DIR + "HEALTHCANADA.csv", index=False)

print(f"HealthCanada dataset saved to {output_path}")


In [ ]:
df.columns

In [ ]:
df.head()

----
# Cut off 1995 & Filtering

In [ ]:
data_filter = "" # "_nonpars" or "_pars" or ""

In [ ]:
DATASET_DIR = "./../datasets"

EMA_path = f"{DATASET_DIR}/EMA.json"
JAPAN_path = f"{DATASET_DIR}/JAPAN.json"
SWISSMEDIC_path = f"{DATASET_DIR}/SWISSMEDIC.json"
AUSTRALIA_path = f"{DATASET_DIR}/AUSTRALIA.json"
FDA_path = f"{DATASET_DIR}/FDA.json"
HEALTHCANADA_path = f"{DATASET_DIR}/HEALTHCANADA.json"

In [ ]:
SAVE_DIR = f"{DATASET_DIR}/1995{data_filter}"
os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_DIR_DATA_ALL = f"{SAVE_DIR}/all_decisions/"
os.makedirs(SAVE_DIR_DATA_ALL, exist_ok=True)
SAVE_DIR_DATA_APPROVED = f"{SAVE_DIR}/approved/"
os.makedirs(SAVE_DIR_DATA_APPROVED, exist_ok=True)

In [ ]:
def prepare_for_analysis(filepath):

    def load_data(file_path):
        """Load data from a JSON file."""
        with open(file_path, 'r', encoding="utf-8") as file:
            data = json.load(file)
        return pd.DataFrame(data)

    def to_lower(df):
        # Columns to exclude from lowercase conversion
        exclude_cols = ['Dataset']
        df_result = df.copy()
        for col in df_result.columns:
            if col not in exclude_cols:
                df_result[col] = df_result[col].map(lambda x: x.lower().strip() if isinstance(x, str) else x)
        return df_result
    
    def harmonize_pharmaceutical_form(df):
        """Standardize pharmaceutical form values across datasets."""
        if 'Pharmaceutical_form' not in df.columns:
            return df
        
        df = df.copy()
        df['Pharmaceutical_form'] = df['Pharmaceutical_form'].str.replace('lyophilized powder', 'lyophilisate', case=False)
        df['Pharmaceutical_form'] = df['Pharmaceutical_form'].str.replace('lyophilized product', 'lyophilisate', case=False)
        df['Pharmaceutical_form'] = df['Pharmaceutical_form'].str.replace('injectable', 'injection', case=False)
        df['Pharmaceutical_form'] = df['Pharmaceutical_form'].str.replace('aqueous injection', 'injection', case=False)
        df['Disease_class(es)'] = df['Disease_class(es)'].str.replace("`diseases of the circulatory system`", "diseases of the circulatory system", case=False)
        df["Drug_class"] = df["Drug_class"].str.replace("antisense oligonucleotide", "peptides and proteins")
        return df

    def remove_invalid_rows(df):
        invalid_rows = df.apply(lambda x: x.astype(str).str.contains("the response is not valid json", case=False, na=False))
        print(f"Total rows with invalid responses: {invalid_rows.any(axis=1).sum()}")
        df = df[~invalid_rows.any(axis=1)]
        return df

    def date_to_int(df, col_name):
        df = df.copy()
        original_len = len(df)

        numeric_years = pd.to_numeric(df[col_name], errors='coerce')
        numeric_years = numeric_years.where(numeric_years.between(1800, 2100))
        dt_years = pd.to_datetime(df[col_name], errors='coerce').dt.year
        df[col_name] = numeric_years.fillna(dt_years).astype('Int64')
        df = df.dropna(subset=[col_name])
        df[col_name] = df[col_name].astype(int)

        final_len = len(df)
        print(f"Original length: {original_len}, Final length after cleaning: {final_len}, "
            f"Percentage kept: {final_len/original_len*100:.2f}%")
        return df

    df = load_data(filepath)
    df = df.transpose()
    df.replace(["not reported", "Not reported", ""], np.nan, inplace=True)
    df = remove_invalid_rows(df)
    df = to_lower(df)
    df = harmonize_pharmaceutical_form(df)
    df = date_to_int(df, 'Decision_year')
    return df

def filter_by_decision_type(df):
    df_new = df.copy()
    if "Decision" not in df_new.columns:
        return df_new
    mask = df_new["Decision"].astype(str).str.lower().str.contains('approved|marketed|conditional marketing authorisation', na=False)
    df_new = df_new[mask]
    return df_new

def filter_by_decision_year(df, min_year=1995):
    """
    Keep only rows where 'Decision_year' is >= min_year.
    If 'Decision_year' column is missing, returns an empty DataFrame with same columns.
    Returns (filtered_df, stats).
    """
    def extract_year_series(ser):
        """Return an Int64 Series of years extracted from a column (numeric year or parsed datetime)."""
        numeric = pd.to_numeric(ser, errors='coerce')
        dt_year = pd.to_datetime(ser, errors='coerce').dt.year
        years = numeric.fillna(dt_year)
        return years.astype("Int64")
    
    stats = {'original_rows': len(df), 'kept': 0, 'dropped': len(df)}
    if 'Decision_year' not in df.columns:
        empty = df.iloc[0:0].copy()
        stats.update({'note': "'Decision_year' column not present; returned empty DataFrame"})
        return empty, stats

    years = extract_year_series(df['Decision_year'])
    mask = years.notna() & (years >= int(min_year))
    mask = mask.fillna(False)
    filtered = df.loc[mask].copy()
    stats['kept'] = int(len(filtered))
    stats['dropped'] = int(stats['original_rows'] - stats['kept'])
    return filtered, stats


In [ ]:
EMA = prepare_for_analysis(EMA_path)
# PMDA = prepare_for_analysis(JAPAN_path)
Swissmedic = prepare_for_analysis(SWISSMEDIC_path)
TGA = prepare_for_analysis(AUSTRALIA_path)
FDA = prepare_for_analysis(FDA_path)
HealthCanada = prepare_for_analysis(HEALTHCANADA_path)

# Combine raw data (used for plots that want every record)
Overall = pd.concat(
    [
        EMA, 
        # PMDA,
        Swissmedic, 
        TGA, 
        FDA, 
        HealthCanada
        ],
    ignore_index=True,
    sort=False
)

DATASETS = {
    'EMA': EMA,
    # 'PMDA': PMDA,
    'Swissmedic': Swissmedic,
    'TGA': TGA,
    'HealthCanada': HealthCanada,
    'FDA': FDA,
    'Overall': Overall
}

# Filtered copies (1995+); overall is built from the already-filtered frames
datasets_orig = {
    'EMA': EMA.copy(),
    # 'PMDA': PMDA.copy(),
    'Swissmedic': Swissmedic.copy(),
    'TGA': TGA.copy(),
    'FDA': FDA.copy(),
    'HealthCanada': HealthCanada.copy()
}

DATASETS_1995 = {}
summary = {}

for dataset_name, dataset_df in datasets_orig.items():
    if dataset_df is None:
        print(f"{dataset_name}: variable not in globals(), skipping.")
        continue
    if not isinstance(dataset_df, pd.DataFrame):
        print(f"{dataset_name}: not a DataFrame (type={type(dataset_df)}), skipping.")
        continue

    filtered_df, stats = filter_by_decision_year(dataset_df, min_year=1995)
    DATASETS_1995[dataset_name] = filtered_df
    summary[dataset_name] = stats
    globals()[f"{dataset_name}_1995"] = filtered_df  # optional, keeps old naming convention

overall_parts = []
for dataset_name, filtered_df in DATASETS_1995.items():
    overall_parts.append(filtered_df.copy())

Overall_1995 = pd.concat(overall_parts, ignore_index=True, sort=False)
DATASETS_1995['Overall'] = Overall_1995
summary['Overall'] = {
    'original_rows': sum(item['kept'] for item in summary.values()),
    'kept': len(Overall_1995),
    'dropped': sum(item['kept'] for item in summary.values()) - len(Overall_1995)
}

for dataset_name, stats in summary.items():
    print(
        f"\n{dataset_name}_1995: original={stats['original_rows']}, "
        f"kept={stats['kept']}, dropped={stats['dropped']}"
    )
    if 'note' in stats:
        print(f"  note: {stats['note']}")


In [ ]:
# Filter for approved/marketed decisions only
DATASETS_1995_approved = {}

for dataset_name, filtered_df in DATASETS_1995.items():
    if dataset_name == 'Overall':
        continue  # Skip Overall, we'll build it from filtered datasets
    
    approved_df = filter_by_decision_type(filtered_df)
    DATASETS_1995_approved[dataset_name] = approved_df
    globals()[f"{dataset_name}_1995_approved"] = approved_df  # optional, keeps naming convention

# Build Overall from already-filtered datasets
overall_approved_parts = []
for dataset_name, approved_df in DATASETS_1995_approved.items():
    overall_approved_parts.append(approved_df.copy())

Overall_1995_approved = pd.concat(overall_approved_parts, ignore_index=True, sort=False)
DATASETS_1995_approved['Overall'] = Overall_1995_approved

# Print summary
print("\n" + "="*60)
print("APPROVED/MARKETED ONLY (1995+):")
print("="*60)
for dataset_name in [
                        'EMA', 
                        # 'PMDA', 
                        'Swissmedic', 
                        'TGA', 
                        'FDA', 
                        'HealthCanada', 
                        'Overall'
                        ]:
    if dataset_name in DATASETS_1995_approved:
        before = len(DATASETS_1995[dataset_name])
        after = len(DATASETS_1995_approved[dataset_name])
        dropped = before - after
        print(f"{dataset_name}_1995_approved: rows={after} (dropped {dropped} from 1995+ dataset)")


In [ ]:
for k,v in DATASETS_1995_approved.items():
    v.to_csv(f"{SAVE_DIR_DATA_APPROVED}{k}.csv", index=False)

for k,v in DATASETS_1995.items():
    v.to_csv(f"{SAVE_DIR_DATA_ALL}{k}.csv", index=False)